In [22]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("rmisra/news-category-dataset")

df = pd.read_json(path + '/News_Category_Dataset_v3.json', lines=True)

print(df.shape)
print("Path to dataset files:", path)

(209527, 6)
Path to dataset files: /home/jens/.cache/kagglehub/datasets/rmisra/news-category-dataset/versions/3


In [23]:
df.describe()

,date
count,209527
mean,2015-04-30 00:44:14.344308
min,2012-01-28 00:00:00
25%,2013-08-10 00:00:00
50%,2015-03-16 00:00:00
75%,2016-11-01 00:00:00
max,2022-09-23 00:00:00


In [24]:
df.sample(5, random_state=42)

,link,headline,category,short_description,authors,date
128310,https://www.huffingtonpost.com/entry/what-if-w...,What If We Were All Family Generation Changers?,IMPACT,"What if, in doing so, we won't just create new...","Matt Murrie, ContributorEdupreneur, Cofounder/...",2014-06-20
139983,https://www.huffingtonpost.comhttp://www.washi...,Firestorm At AOL Over Employee Benefit Cuts,BUSINESS,It should have been a glorious week for AOL ch...,,2014-02-08
42339,https://www.huffingtonpost.com/entry/time-runs...,Dakota Access Protesters Arrested As Deadline ...,POLITICS,A few protesters who refused to leave remained...,"Michael McLaughlin & Josh Morgan, The Huffingt...",2017-02-22
131494,https://www.huffingtonpost.com/entry/one-glimp...,One Glimpse Of These Baby Kit Foxes And You'll...,GREEN,,,2014-05-14
163649,https://www.huffingtonpost.com/entry/mens-swea...,"Mens' Sweat Pheromone, Androstadienone, Influe...",SCIENCE,Scientists didn't know if humans played that g...,Melissa Cronin,2013-06-02


In [25]:
print("Number of unique categories:", df['category'].nunique())
print("\nCategory counts:")
print(df['category'].value_counts())

Number of unique categories: 42

Category counts:
category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATIN

In [26]:
df['combined_text'] = df['headline'] + ' ' + df['short_description']

print(df.shape)
clean_df = df[['combined_text', 'category']].copy()
clean_df.dropna(inplace=True)
clean_df.drop_duplicates(inplace=True)
print(clean_df.shape)
clean_df.head(5)

(209527, 7)
(209056, 2)


,combined_text,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [27]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    sanitized_text = re.sub(r'[^a-zA-Z\s]', '', text)
    sanitized_text = sanitized_text.lower()
    split_text = sanitized_text.split()
    text_without_stopwords = [word for word in split_text if word not in stop_words]
    lemmatized_text = [lemmatizer.lemmatize(word) for word in text_without_stopwords]
    return ' '.join(lemmatized_text)

clean_df['combined_text'] = clean_df['combined_text'].apply(preprocess_text)
clean_df.head(5)

[nltk_data] Downloading package stopwords to /home/jens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jens/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,combined_text,category
0,million american roll sleeve omicrontargeted c...,U.S. NEWS
1,american airline flyer charged banned life pun...,U.S. NEWS
2,funniest tweet cat dog week sept dog dont unde...,COMEDY
3,funniest tweet parent week sept accidentally p...,PARENTING
4,woman called cop black birdwatcher loses lawsu...,U.S. NEWS


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(clean_df['category'])

x = clean_df['combined_text'].values
y = y_encoded

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("Classes:", label_encoder.classes_)
print("Train size:", len(x_train))
print("Test size:", len(x_test))

Classes: ['ARTS' 'ARTS & CULTURE' 'BLACK VOICES' 'BUSINESS' 'COLLEGE' 'COMEDY'
 'CRIME' 'CULTURE & ARTS' 'DIVORCE' 'EDUCATION' 'ENTERTAINMENT'
 'ENVIRONMENT' 'FIFTY' 'FOOD & DRINK' 'GOOD NEWS' 'GREEN' 'HEALTHY LIVING'
 'HOME & LIVING' 'IMPACT' 'LATINO VOICES' 'MEDIA' 'MONEY' 'PARENTING'
 'PARENTS' 'POLITICS' 'QUEER VOICES' 'RELIGION' 'SCIENCE' 'SPORTS' 'STYLE'
 'STYLE & BEAUTY' 'TASTE' 'TECH' 'THE WORLDPOST' 'TRAVEL' 'U.S. NEWS'
 'WEDDINGS' 'WEIRD NEWS' 'WELLNESS' 'WOMEN' 'WORLD NEWS' 'WORLDPOST']
Train size: 167244
Test size: 41812


In [29]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


In [30]:
VOCAB_SIZE  = 50000   # Maximum vocabulary size
MAX_LEN     = 64      # Maximum sequence length
EMBED_DIM   = 128     # Word embedding dimensions
HIDDEN_DIM  = 256     # RNN hidden state size
NUM_LAYERS  = 2       # Stacked RNN layers
BATCH_SIZE  = 64
EPOCHS      = 5
LEARNING_RATE = 1e-3
NUM_CLASSES = len(label_encoder.classes_)

print("Number of classes:", NUM_CLASSES)

Number of classes: 42


In [31]:
# Build vocabulary from training data
# Special tokens: <PAD> = 0, <UNK> = 1
PAD_IDX = 0
UNK_IDX = 1

all_words = [word for text in x_train for word in text.split()]
word_counts = Counter(all_words)
vocab = ['<PAD>', '<UNK>'] + [word for word, _ in word_counts.most_common(VOCAB_SIZE - 2)]
word2idx = {word: idx for idx, word in enumerate(vocab)}

print("Vocabulary size:", len(vocab))

Vocabulary size: 50000


In [32]:
def encode_text(text, word2idx, max_len):
    tokens = text.split()[:max_len]
    ids = [word2idx.get(word, UNK_IDX) for word in tokens]
    # Pad or truncate to max_len
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids


class NewsDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
        self.texts   = texts
        self.labels  = labels
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode_text(self.texts[idx], self.word2idx, self.max_len)
        return {
            'input_ids': torch.tensor(ids, dtype=torch.long),
            'label':     torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_dataset = NewsDataset(x_train, y_train, word2idx, MAX_LEN)
test_dataset  = NewsDataset(x_test,  y_test,  word2idx, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Test batches: ", len(test_loader))

Train batches: 2614
Test batches:  654


In [33]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, pad_idx):
        super(RNNClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # Bidirectional LSTM reads the sequence forwards and backwards
        self.rnn = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.dropout    = nn.Dropout(0.3)
        # *2 because bidirectional
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids):
        # input_ids: (batch, seq_len)
        embedded = self.dropout(self.embedding(input_ids))   # (batch, seq_len, embed_dim)
        output, (hidden, _) = self.rnn(embedded)             # hidden: (num_layers*2, batch, hidden_dim)

        # Concatenate the final forward and backward hidden states
        hidden_fwd = hidden[-2]   # last forward layer
        hidden_bwd = hidden[-1]   # last backward layer
        combined   = torch.cat([hidden_fwd, hidden_bwd], dim=1)  # (batch, hidden_dim*2)

        return self.classifier(self.dropout(combined))       # (batch, num_classes)


model = RNNClassifier(
    vocab_size  = len(vocab),
    embed_dim   = EMBED_DIM,
    hidden_dim  = HIDDEN_DIM,
    num_layers  = NUM_LAYERS,
    num_classes = NUM_CLASSES,
    pad_idx     = PAD_IDX
)
model = model.to(device)
print("Model loaded and moved to", device)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

Model loaded and moved to cuda
RNNClassifier(
  (embedding): Embedding(50000, 128, padding_idx=0)
  (rnn): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=512, out_features=42, bias=True)
)

Trainable parameters: 8,789,034


In [34]:
optimizer  = Adam(model.parameters(), lr=LEARNING_RATE)
criterion  = nn.CrossEntropyLoss()

print("Optimizer and loss function ready.")

Optimizer and loss function ready.


In [36]:
import time

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    start = time.time()

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        labels    = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids)
        loss    = criterion(outputs, labels)
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if (step + 1) % 100 == 0:
            elapsed = time.time() - start
            print(f"  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | "
                  f"Loss: {total_loss / (step+1):.4f} | Elapsed: {elapsed:.1f}s")

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} complete — Avg Loss: {avg_loss:.4f}\n")

# Took around 30 minutes on CPU (AMD Ryzen 9 7900X 12-Core Processor) for 5 epochs. With GPU, it should be much faster.
# Using py3.11 4 minutes 11 seconds (NVIDIA 1080-ti) for 5 epochs

  Epoch 1 | Step 100/2614 | Loss: 3.2665 | Elapsed: 2.1s
  Epoch 1 | Step 200/2614 | Loss: 3.1213 | Elapsed: 4.1s
  Epoch 1 | Step 300/2614 | Loss: 3.0125 | Elapsed: 6.1s
  Epoch 1 | Step 400/2614 | Loss: 2.9305 | Elapsed: 8.0s
  Epoch 1 | Step 500/2614 | Loss: 2.8589 | Elapsed: 9.9s
  Epoch 1 | Step 600/2614 | Loss: 2.7892 | Elapsed: 11.9s
  Epoch 1 | Step 700/2614 | Loss: 2.7340 | Elapsed: 13.8s
  Epoch 1 | Step 800/2614 | Loss: 2.6854 | Elapsed: 15.8s
  Epoch 1 | Step 900/2614 | Loss: 2.6434 | Elapsed: 17.7s
  Epoch 1 | Step 1000/2614 | Loss: 2.6038 | Elapsed: 19.7s
  Epoch 1 | Step 1100/2614 | Loss: 2.5695 | Elapsed: 21.7s
  Epoch 1 | Step 1200/2614 | Loss: 2.5355 | Elapsed: 23.7s
  Epoch 1 | Step 1300/2614 | Loss: 2.5047 | Elapsed: 25.7s
  Epoch 1 | Step 1400/2614 | Loss: 2.4736 | Elapsed: 27.8s
  Epoch 1 | Step 1500/2614 | Loss: 2.4476 | Elapsed: 29.8s
  Epoch 1 | Step 1600/2614 | Loss: 2.4229 | Elapsed: 31.8s
  Epoch 1 | Step 1700/2614 | Loss: 2.3994 | Elapsed: 33.9s
  Epoch 1 |

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids']
        labels    = batch['label']

        outputs = model(input_ids)
        preds   = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

y_pred      = all_preds
y_test_eval = all_labels

accuracy  = accuracy_score(y_test_eval, y_pred)
precision = precision_score(y_test_eval, y_pred, average='weighted')
recall    = recall_score(y_test_eval, y_pred, average='weighted')
f1        = f1_score(y_test_eval, y_pred, average='weighted')

print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-score:  {f1 * 100:.2f}%")

print("\nClassification Report:\n",
      classification_report(y_test_eval, y_pred, target_names=label_encoder.classes_))

In [ ]:
import joblib

torch.save(model.state_dict(), f'category_rnn_model_acc_{accuracy * 100:.2f}%.pt')
print("RNN model saved.")

joblib.dump(label_encoder, 'category_rnn_label_encoder.pkl')
joblib.dump(word2idx, 'category_rnn_word2idx.pkl')
print("Label encoder and vocabulary saved.")

In [ ]:
def predict_category(text, model, word2idx, label_encoder, max_len=64):
    import re
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    stop_words  = set(stopwords.words('english'))
    lemmatizer  = WordNetLemmatizer()
    clean = re.sub(r'[^a-zA-Z\s]', '', text).lower().split()
    clean = [lemmatizer.lemmatize(w) for w in clean if w not in stop_words]
    clean = ' '.join(clean)

    model.eval()
    ids = encode_text(clean, word2idx, max_len)
    input_ids = torch.tensor([ids], dtype=torch.long)

    with torch.no_grad():
        output = model(input_ids)
        pred   = torch.argmax(output, dim=1).item()

    return label_encoder.inverse_transform([pred])[0]

examples = [
    "Stock market hits record high amid strong earnings reports",
    "New study reveals benefits of Mediterranean diet for heart health",
    "Local football team wins championship in overtime thriller"
]

for text in examples:
    category = predict_category(text, model, word2idx, label_encoder)
    print(f"Text:     {text}")
    print(f"Category: {category}\n")